# High-Bandwidth Disaster Dataset-to-Drive `.tar` Stager (Colab -> Drive API v3 Direct Push)

Tối đa hoá băng thông Google Cloud của Colab (**chạy tốt trên `CPU + High-RAM` để tiết kiệm Compute Units, hoặc `G4` nếu muốn cổng mạng `50 Gbps` tối đa**) để kéo các bộ dữ liệu thảm hoạ, làm sạch `0%` trùng lặp, đóng `.tar` không nén, đẩy thẳng lên **Google Drive API v3 (`64 MB` resumable chunks)** và **tự động xuất link tải dùng ngay**:

- **Hugging Face (`DisasterM3`, `DisasterVQA-Incidents`):** Dùng **Git-LFS Batch API (`us.aws.cdn.hf.co`) + `aria2c -j 32` + `SHA-256`**, vượt hoàn toàn giới hạn `/resolve/`.
- **Google Drive API v3 Direct Upload:** Ghi trực tiếp qua API Google thay vì FUSE mount (nhanh gấp 3–5 lần, không bị lỗi đồng bộ ngầm khi ngắt phiên), tự động bật chia sẻ công khai (nếu chọn `MAKE_SHAREABLE_LINKS = True`) và in sẵn 3 biến `*_ARCHIVE_URL` ở Cell cuối.


In [ ]:
#@title Dataset Staging Configuration { display-mode: "form" }
DRIVE_FOLDER_NAME = "mnarm_iccv_datasets"  # @param {type:"string"}
MAKE_SHAREABLE_LINKS = True  # @param {type:"boolean"}
STAGE_DMD = True  # @param {type:"boolean"}
STAGE_INCIDENTS_VQA = True  # @param {type:"boolean"}
STAGE_DISASTER_M3 = True  # @param {type:"boolean"}
STAGE_CRISISMMD = False  # @param {type:"boolean"}
OVERWRITE_EXISTING = False  # @param {type:"boolean"}
CUSTOM_DMD_URL = ""  # @param {type:"string"}
CUSTOM_INCIDENTS_ZIP_URL = ""  # @param {type:"string"}
CUSTOM_CRISISMMD_TAR_URL = ""  # @param {type:"string"}


In [ ]:
import os, shutil, subprocess
from pathlib import Path
from google.colab import auth
from googleapiclient.discovery import build

if shutil.which("aria2c") is None and shutil.which("apt-get"):
    subprocess.run(["apt-get", "-qq", "update"], check=False)
    subprocess.run(["apt-get", "-qq", "install", "-y", "aria2", "pigz", "unzip"], check=False)

auth.authenticate_user()
DRIVE_SVC = build("drive", "v3", cache_discovery=False)

folder_name = str(DRIVE_FOLDER_NAME).strip() or "mnarm_iccv_datasets"
q_folder = f"name = '{folder_name}' and mimeType = 'application/vnd.google-apps.folder' and 'root' in parents and trashed = false"
found = DRIVE_SVC.files().list(q=q_folder, fields="files(id, name)").execute().get("files", [])
if found:
    DRIVE_FOLDER_ID = found[0]["id"]
else:
    meta = {"name": folder_name, "mimeType": "application/vnd.google-apps.folder", "parents": ["root"]}
    DRIVE_FOLDER_ID = DRIVE_SVC.files().create(body=meta, fields="id").execute()["id"]

shm_free_gb = shutil.disk_usage("/dev/shm").free / 1e9 if Path("/dev/shm").exists() else 0.0
root_free_gb = shutil.disk_usage("/content").free / 1e9
STAGE_ROOT = Path("/dev/shm/mnarm_stage_tmp") if shm_free_gb >= 10.0 else Path("/content/mnarm_stage_tmp")
STAGE_ROOT.mkdir(parents=True, exist_ok=True)
print(f"[READY] Drive folder='{folder_name}' (id={DRIVE_FOLDER_ID}) | STAGE_ROOT={STAGE_ROOT} | /dev/shm={shm_free_gb:.1f}GB | /content={root_free_gb:.1f}GB", flush=True)


In [ ]:
import concurrent.futures, hashlib, json, os, shutil, subprocess, time, urllib.parse, urllib.request
from pathlib import Path
from googleapiclient.http import MediaFileUpload

try:
    from google.colab import userdata as colab_userdata
    if not os.environ.get("HF_TOKEN"):
        tok_u = (colab_userdata.get("HF_TOKEN") or "").strip()
        if tok_u:
            os.environ["HF_TOKEN"] = tok_u
except Exception:
    pass

HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
INCIDENTS_REV = "f341ce34814a93bad8abfe73da898a309116a59e"
M3_REV = "1da6da73fad6bd100454c099edd0cee972823810"
DMD_UCI_URL = "https://archive.ics.uci.edu/static/public/456/multimodal+damage+identification+for+humanitarian+computing.zip"

def _sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as fh:
        while True:
            buf = fh.read(16 * 1024 * 1024)
            if not buf:
                break
            h.update(buf)
    return h.hexdigest()

def _find_drive_file(name: str) -> dict | None:
    q = f"name = '{name}' and '{DRIVE_FOLDER_ID}' in parents and trashed = false"
    res = DRIVE_SVC.files().list(q=q, fields="files(id, name, size)").execute().get("files", [])
    return res[0] if res else None

def _share_if_requested(file_id: str) -> None:
    if MAKE_SHAREABLE_LINKS:
        DRIVE_SVC.permissions().create(fileId=file_id, body={"type": "anyone", "role": "reader"}, fields="id").execute()

def _lfs_batch_aria2(repo_id: str, rev: str, rel_paths: list[str], dest_dir: Path) -> None:
    dest_dir.mkdir(parents=True, exist_ok=True)
    req_hdrs = {"User-Agent": "Mozilla/5.0"}
    if HF_TOKEN:
        req_hdrs["Authorization"] = f"Bearer {HF_TOKEN}"
    t0 = time.perf_counter()
    with urllib.request.urlopen(urllib.request.Request(f"https://huggingface.co/api/datasets/{repo_id}?blobs=1", headers=req_hdrs), timeout=30) as r:
        meta_map = {str(s["rfilename"]): s for s in json.loads(r.read()).get("siblings", [])}

    lfs_items, direct_items = [], []
    for rel in sorted(set(r.replace("\\", "/").lstrip("/") for r in rel_paths)):
        dst = dest_dir / rel
        meta = meta_map.get(rel, {})
        lfs = meta.get("lfs")
        if isinstance(lfs, dict) and lfs.get("sha256") and lfs.get("size"):
            oid, sz = str(lfs["sha256"]), int(lfs["size"])
            if dst.is_file() and dst.stat().st_size == sz:
                continue
            lfs_items.append((rel, oid, sz, dst))
        else:
            if dst.is_file() and meta.get("size") and dst.stat().st_size == int(meta["size"]):
                continue
            direct_items.append((f"https://huggingface.co/datasets/{repo_id}/resolve/{rev}/{urllib.parse.quote(rel, safe='/')}", {"Authorization": f"Bearer {HF_TOKEN}"} if HF_TOKEN else {}, "", dst))

    if lfs_items:
        uniq_objs = list(dict.fromkeys((o, s) for _, o, s, _ in lfs_items))
        actions = {}
        lfs_hdrs = {"Accept": "application/vnd.git-lfs+json", "Content-Type": "application/vnd.git-lfs+json", "User-Agent": "git-lfs/3.4.0"}
        if HF_TOKEN:
            lfs_hdrs["Authorization"] = f"Bearer {HF_TOKEN}"
        for off in range(0, len(uniq_objs), 100):
            body = json.dumps({"operation": "download", "transfers": ["basic"], "objects": [{"oid": o, "size": s} for o, s in uniq_objs[off:off+100]]}).encode("utf8")
            with urllib.request.urlopen(urllib.request.Request(f"https://huggingface.co/datasets/{repo_id}.git/info/lfs/objects/batch", data=body, headers=lfs_hdrs, method="POST"), timeout=30) as r:
                for obj in json.loads(r.read()).get("objects", []):
                    dl = obj.get("actions", {}).get("download", {})
                    if dl.get("href"):
                        actions[str(obj["oid"])] = (str(dl["href"]), {str(k): str(v) for k, v in dl.get("header", {}).items()})
        for rel, oid, _, dst in lfs_items:
            if oid in actions:
                direct_items.append((actions[oid][0], actions[oid][1], oid, dst))
            else:
                direct_items.append((f"https://huggingface.co/datasets/{repo_id}/resolve/{rev}/{urllib.parse.quote(rel, safe='/')}", {"Authorization": f"Bearer {HF_TOKEN}"} if HF_TOKEN else {}, "", dst))

    if direct_items:
        mf_lines = []
        for url, hmap, oid, dst in direct_items:
            dst.parent.mkdir(parents=True, exist_ok=True)
            mf_lines.extend([url, f"  dir={dst.parent}", f"  out={dst.name}"])
            for hk, hv in hmap.items():
                mf_lines.append(f"  header={hk}: {hv}")
            if oid:
                mf_lines.append(f"  checksum=sha-256={oid}")
        mf = STAGE_ROOT / f"aria2_{repo_id.replace('/', '_')}.txt"
        mf.write_text("\n".join(mf_lines) + "\n", encoding="utf8")
        subprocess.run(["aria2c", f"--input-file={mf}", "-j", "32", "-x", "4", "-s", "4", "-k", "1M", "--check-integrity=true", "--retry-wait=3", "--max-tries=6", "--auto-file-renaming=false", "--allow-overwrite=true", "--console-log-level=warn", "--summary-interval=10"], check=False)
        mf.unlink(missing_ok=True)

    missing = [it for it in direct_items if not it[3].is_file()]
    if missing:
        def _dl_one(it):
            url, hmap, _, dst = it
            dst.parent.mkdir(parents=True, exist_ok=True)
            with urllib.request.urlopen(urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0", **hmap}), timeout=60) as r:
                dst.write_bytes(r.read())
        with concurrent.futures.ThreadPoolExecutor(max_workers=12) as pool:
            for f in concurrent.futures.as_completed([pool.submit(_dl_one, m) for m in missing]):
                f.result()
    tot_mb = sum(dst.stat().st_size for _, _, _, dst in direct_items if dst.is_file()) / 1e6
    dt = max(time.perf_counter() - t0, 0.01)
    print(f"  [CDN] {repo_id}: {len(direct_items)} files ({tot_mb:.1f} MB) in {dt:.1f}s ({tot_mb/dt:.1f} MB/s)", flush=True)

def _pack_and_upload_drive_api(src_dir: Path, tar_name: str) -> dict:
    local_tar = STAGE_ROOT / tar_name
    local_tar.unlink(missing_ok=True)
    t0 = time.perf_counter()
    subprocess.run(["tar", "-cf", str(local_tar), "-C", str(src_dir), "."], check=True)
    sz = local_tar.stat().st_size
    dt_pack = max(time.perf_counter() - t0, 0.01)
    digest = _sha256_file(local_tar)
    shutil.rmtree(src_dir, ignore_errors=True)

    existing = _find_drive_file(tar_name)
    if existing:
        DRIVE_SVC.files().delete(fileId=existing["id"]).execute()

    t1 = time.perf_counter()
    media = MediaFileUpload(str(local_tar), mimetype="application/x-tar", chunksize=64 * 1024 * 1024, resumable=True)
    req = DRIVE_SVC.files().create(body={"name": tar_name, "parents": [DRIVE_FOLDER_ID]}, media_body=media, fields="id, size, webViewLink")
    resp = None
    while resp is None:
        status, resp = req.next_chunk()
        if status:
            print(f"  [DRIVE API] {tar_name}: {status.progress()*100:.0f}%", flush=True)
    dt_up = max(time.perf_counter() - t1, 0.01)
    fid = resp["id"]
    _share_if_requested(fid)
    local_tar.unlink(missing_ok=True)
    uc_url = f"https://drive.google.com/uc?id={fid}&export=download"
    view_url = resp.get("webViewLink", f"https://drive.google.com/file/d/{fid}/view?usp=sharing")
    print(f"[DONE] {tar_name}: {sz/1e6:.1f} MB | tar={(sz/1e6)/dt_pack:.0f} MB/s | api_up={(sz/1e6)/dt_up:.1f} MB/s | id={fid}", flush=True)
    return {"tar": tar_name, "file_id": fid, "bytes": sz, "sha256": digest, "direct_url": uc_url, "view_url": view_url}

RESULTS = {}

def _maybe_skip(ds_key: str, tar_name: str) -> bool:
    ex = _find_drive_file(tar_name)
    if ex and not OVERWRITE_EXISTING and int(ex.get("size", 0)) > 1_000_000:
        fid = ex["id"]
        _share_if_requested(fid)
        RESULTS[ds_key] = {
            "tar": tar_name, "file_id": fid, "bytes": int(ex["size"]), "sha256": "existing",
            "direct_url": f"https://drive.google.com/uc?id={fid}&export=download",
            "view_url": f"https://drive.google.com/file/d/{fid}/view?usp=sharing",
        }
        print(f"[SKIP EXISTING] {tar_name} ({int(ex['size'])/1e6:.1f} MB) id={fid}", flush=True)
        return True
    return False

if STAGE_DMD and not _maybe_skip("dmd", "dmd.tar"):
    print("\n=== [1] STAGING DMD (UCI #456) ===", flush=True)
    dmd_work = STAGE_ROOT / "dmd_work"
    shutil.rmtree(dmd_work, ignore_errors=True); dmd_work.mkdir(parents=True, exist_ok=True)
    dmd_zip = STAGE_ROOT / "dmd_raw.zip"
    subprocess.run(["curl", "-fL", "--retry", "4", str(CUSTOM_DMD_URL).strip() or DMD_UCI_URL, "-o", str(dmd_zip)], check=True)
    subprocess.run(["unzip", "-q", "-o", str(dmd_zip), "-d", str(dmd_work)], check=True); dmd_zip.unlink(missing_ok=True)
    for sub_zip in list(dmd_work.rglob("*.zip")):
        subprocess.run(["unzip", "-q", "-o", str(sub_zip), "-d", str(dmd_work)], check=True); sub_zip.unlink(missing_ok=True)
    RESULTS["dmd"] = _pack_and_upload_drive_api(dmd_work, "dmd.tar")

if STAGE_INCIDENTS_VQA and not _maybe_skip("incidents_vqa", "incidents_vqa.tar"):
    print("\n=== [2] STAGING DisasterVQA-Incidents (Incidents1M-only, 0% CrisisMMD) ===", flush=True)
    vq_work = STAGE_ROOT / "incidents_vqa_work"
    shutil.rmtree(vq_work, ignore_errors=True); vq_work.mkdir(parents=True, exist_ok=True)
    if str(CUSTOM_INCIDENTS_ZIP_URL).strip():
        vzip = STAGE_ROOT / "vqa_custom.zip"
        subprocess.run(["aria2c", "-x16", "-s16", "-k1M", str(CUSTOM_INCIDENTS_ZIP_URL).strip(), "-d", str(STAGE_ROOT), "-o", vzip.name], check=True)
        subprocess.run(["unzip", "-q", "-o", str(vzip), "-d", str(vq_work)], check=True); vzip.unlink(missing_ok=True)
        for drop_n in ("CrisisMMD", "MEDIC"):
            for d in vq_work.rglob(drop_n):
                if d.is_dir():
                    shutil.rmtree(d, ignore_errors=True)
    else:
        _lfs_batch_aria2("QCRI/DisasterVQA", INCIDENTS_REV, ["disastervqa_annotations.jsonl", "taxonomy.json"], vq_work)
        inc_rels = sorted({
            str(json.loads(ln)["image_path"]).replace("\\", "/").lstrip("/")
            for ln in (vq_work / "disastervqa_annotations.jsonl").read_text(encoding="utf8").splitlines()
            if ln.strip() and json.loads(ln).get("dataset_source") == "incidents1m"
        })
        _lfs_batch_aria2("QCRI/DisasterVQA", INCIDENTS_REV, inc_rels, vq_work)
    RESULTS["incidents_vqa"] = _pack_and_upload_drive_api(vq_work, "incidents_vqa.tar")

if STAGE_DISASTER_M3 and not _maybe_skip("disaster_m3", "disaster_m3.tar"):
    print("\n=== [3] STAGING DisasterM3 (NeurIPS'25 Optical + SAR, 2363 tiles via LFS Batch CDN) ===", flush=True)
    m3_work = STAGE_ROOT / "disaster_m3_work"
    shutil.rmtree(m3_work, ignore_errors=True); m3_work.mkdir(parents=True, exist_ok=True)
    _lfs_batch_aria2("Kingdrone-Junjue/DisasterM3", M3_REV, ["DisasterM3_Bench/benchmark_release.json"], m3_work)
    rows_m3 = json.loads((m3_work / "DisasterM3_Bench/benchmark_release.json").read_text(encoding="utf8"))
    m3_rels = sorted({
        "DisasterM3_Bench/" + str(r["post_image_path"]).replace("\\", "/").lstrip("/")
        for r in rows_m3 if r.get("task") == "Disaster Report" and r.get("post_image_path")
    })
    _lfs_batch_aria2("Kingdrone-Junjue/DisasterM3", M3_REV, m3_rels, m3_work)
    RESULTS["disaster_m3"] = _pack_and_upload_drive_api(m3_work, "disaster_m3.tar")

if STAGE_CRISISMMD and not _maybe_skip("crisismmd", "crisismmd_v2.tar"):
    print("\n=== [OPT] STAGING CrisisMMD_v2 ===", flush=True)
    cm_work = STAGE_ROOT / "crisismmd_work"
    shutil.rmtree(cm_work, ignore_errors=True); cm_work.mkdir(parents=True, exist_ok=True)
    cm_tgz = STAGE_ROOT / "CrisisMMD_v2.0.tar.gz"
    subprocess.run(["aria2c", "-x16", "-s16", "-k1M", str(CUSTOM_CRISISMMD_TAR_URL).strip() or "https://crisisnlp.qcri.org/data/crisismmd/CrisisMMD_v2.0.tar.gz", "-d", str(STAGE_ROOT), "-o", cm_tgz.name], check=True)
    subprocess.run(["tar", "-I", "pigz", "-xf", str(cm_tgz), "-C", str(cm_work)], check=True); cm_tgz.unlink(missing_ok=True)
    RESULTS["crisismmd"] = _pack_and_upload_drive_api(cm_work, "crisismmd_v2.tar")

man_local = STAGE_ROOT / "DATASET_ARCHIVES_MANIFEST.json"
man_local.write_text(json.dumps(RESULTS, indent=2), encoding="utf8")
old_m = _find_drive_file(man_local.name)
if old_m:
    DRIVE_SVC.files().delete(fileId=old_m["id"]).execute()
DRIVE_SVC.files().create(body={"name": man_local.name, "parents": [DRIVE_FOLDER_ID]}, media_body=MediaFileUpload(str(man_local), mimetype="application/json"), fields="id").execute()
man_local.unlink(missing_ok=True)
print("[MANIFEST UPLOADED TO DRIVE]", flush=True)


In [ ]:
print("=== COPY-PASTE DIRECT DOWNLOAD LINKS FOR ICCV_SameBank_ZeroMNAR_ColabWeb.ipynb ===")
for k, var in (("dmd", "DMD_ARCHIVE_URL"), ("incidents_vqa", "INCIDENTS_VQA_ARCHIVE_URL"), ("disaster_m3", "DISASTER_M3_ARCHIVE_URL"), ("crisismmd", "CRISISMMA_TAR_URL")):
    if k in RESULTS:
        print(f'{var} = "{RESULTS[k]["direct_url"]}"  # ({RESULTS[k]["bytes"]/1e6:.1f} MB) view: {RESULTS[k]["view_url"]}')
